## 1. Introduction<br>

### About the SUPPORT2 Dataset
The SUPPORT2 dataset is comprised of data for 9105 critically ill patients in the United States between 1989-1991 and 1992-1994. The data in this dataset was originally collected for SUPPORT (Study to Understand Prognoses Preferences Outcomes and Risks of Treatment) phase II, an observational clinical trial that followed patients via medial record over the course of 2 year, or until death. It contains information about patient demographics, physiological values (i.e., blood test results), and disease information collected at the time of study entry (Day 3). The goal of this study was to predict the 2- and 6-month survival rates of critically ill patients using the information collected at Day 3.

### The Problem
In addition to improving clinical outcomes, end-of-life research like SUPPORT phase II aims to provide patients and their families with the information to make more informed decisions near the end of life. At 2 months after study entry, patients are assigned a Functional Disability Score at Month 2 (`sfdm2`). This is a validated 5-point scale for describing the level of mental and physical disability, including quality of life, at 2 months after study entry, where lower scores indicate better clinical outcomes and a higher qualty of life at the 2-month mark, whereas a higher score indicates a higher level of disability at Month 2 (note: a score of 5 means the patient has died prior to the 2-month mark, and is therefore considered a poor clinical outcome). Using the data collected at study entry, the goal is to predict the `sfdm2` score for a patient after 2 months to support earlier and better-decision making. 

**Note**: The `sfdm2` score is a measure that aggregates information from many different sources (i.e., patients themselves, caregivers, surrogates, and physicians), and is therefore a reliable descriptor of patients' level of functional disability.  

## 2. Methods<br>
#### 2.1 EDA <br>
***********TO DO***********

#### 2.2 Data Pre-Processing<br>
- Pipeline:
1. Encode categorical data
2. Training/Test split
3. Scale splits separately
4. Impute splits separately

- Encoding Methods<br>
Our initial EDA demonstrated low cardinality for all categorical features. To determine their respective encoding methods, we divided our categorical features into non-ordinal vs ordinal. Non-ordinal features such as race and sex where one hot encoded using pandas 'get_dummies'. Ordinal features were ordinally encoded manually via structured enumeration. The exact order, or ranking, of the categorical variables was specified during the encoding in order to maintain the natural, meaningful order between categories. 


- Scaling Methods<br>
We selected Standard Scaler to ensure features with inherently larger numerical magnitudes do not dominate. Values for biological data, including blood test values in our case, operate on diverse scales (i.e., concentrations vs counts). These values tend to have very different magnitudes; some numerical values end up much larger than others. Standard Scaler helps prevent these larger values from dominating, as it transforms the data to a standard distribution with a mean of 0 and standard deviation of 1. 


- Imputation Methods<br>
Initial data-processing showed that 40% of features contained NaNs. We addressed the missing data by imputing using the K-nearest neighbors method. This was our preferred method given the large percentage of missing data in our dataset. Because K-Nearest Neighbors assumes similar data points exist near eachother, it was a better alternative to other imputation methods, like mean or mode imputation, where the value calculated for the mean or mode can be heavily affected by the fraction of missing data. K-NN imputation, on the other hand, makes a prediction based on its similarity, as a function of its distance, from its neighbors. 


#### 2.3 Implemented Models<br>
- Multinomial Logistic Regression ***********TO DO***********
- Leiden Clustering ***********TO DO***********
- Gaussian Mixture Models ***********TO DO***********
- Naive Bayes ***********TO DO***********

#### 2.4 Evaluation Methods<br>
- Accuracy: To determine the accuracy of the predicted vs true labels for the training and test sets, the accuracy of supervised model’s predictions were calculated using sklearn’s accuracy_score.

- Visualization
    1. tSNE: In order to understand the resulting accuracy of our predictions, we used t-distributed stochastic neighbor embedding (t-SNE) to project our highly dimensional data into a lower dimensional space. This allowed us to observe our data on a 2D graph to observe how the dataset’s predefined classes are clustered, and if the clusters are well separated or highly overlapping. These observations were made on both the true and predicted labels of our supervised model, which helped us to identify why a model may have assigned a point to a specific cluster and to inform our fine-tune our feature pre-processing.

    2. UMAP (Leiden): For our unsupervised Leiden clustering, UMAP was the chosen visualization technique because it’s constructed using k-nearest neighbors and maintains the local and global organization of the data. This functionality of UMAP is similar to that of Leiden’s underlying architecture, and allows us to observe the clusters and their labels while maintaining a consistent and representative 2D projection between both UMAP and Leiden.

- Entropy: ***********TO DO***********


## 3. Implementation


In [3]:
# insert libraries here
# standard
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# data pre-proccessing
from sklearn.preprocessing import LabelEncoder, StandardScaler

# classification models
from sklearn.mixture import GaussianMixture
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import *
import leidenalg

# dimensionality reduction
from sklearn.manifold import TSNE

# metric tools
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from scipy.stats import entropy
import networkx as nx 
from sklearn.neighbors import NearestNeighbors
import igraph as ig 
import umap.umap_ as umap

# SUPPORT2
from ucimlrepo import fetch_ucirepo
support2 = fetch_ucirepo(id=880)

#### Data Pre-Processing

In [ ]:
'''Loading the data'''
def get_dataset():
    # data (as pandas dataframes)
    features = support2.data.features
    targets = support2.data.targets

    # join together as 1 df
    df = features.join(targets)
    df = df.drop(columns=['death', 'hospdead', 'adlp', 'adls', 'scoma', 'totmcst', 'totcst', 'sps', 'aps', 'hday', \
                        'adlsc', 'prg2m', 'prg6m', 'charges', 'dzgroup', 'dnr', 'dnrday', 'urine',
                        'surv2m', 'surv6m']) # added surv2m, surv6m to drop; prevent data leakage

    # drop rows where target = NaN
    print(f'Before dropping NaNs in target: {df.shape[0]}')
    df = df.dropna(subset=['sfdm2'])
    print(f'After dropping NaNs in target: {df.shape[0]}')

    return

In [ ]:
''' Combining classes, encoding data '''
def get_encodings(df: pd.DataFrame):
    ''' combines target classes to improve class imbalance, encodes categorical features via dummy or ordinal encoding'''
    # isolate categorical features
    df_to_encode = df.copy()
    numerical_feats = df.select_dtypes(include=['number']).columns.tolist() # determine numerical columns
    df_to_encode = df_to_encode.drop(numerical_feats, axis=1) # drop numerical columns

    # ------------ DUMMY ENCODING -----------
    dum_cols = ['sex', 'dzclass', 'race', 'ca']
    df_to_encode = pd.get_dummies(df_to_encode, columns=dum_cols, dtype=int)

    # ------------ ORDINAL ENCODING -----------
    income_order = {
        'under $11k': 0,
        '$11-$25k': 1,
        '$25-$50k': 2,
        '>$50k': 3
    }

    df_to_encode['income'] = df_to_encode['income'].map(income_order)

    target_order = {
        'no(M2 and SIP pres)': 0, # 0 = healthy
        'adl>=4 (>=5 if sur)': 1, # 1 = moderate disability
        'SIP>=30': 1,
        'Coma or Intub': 1,
        '<2 mo. follow-up': 2     # 2 = death
    }

    df_to_encode['sfdm2'] = df_to_encode['sfdm2'].map(target_order)

    # recombine dataframe
    df = pd.concat([df_to_encode, df[numerical_feats]], axis=1)

    # examine class imbalance
    counts = df['sfdm2'].value_counts()
    plt.bar(counts.index, counts.values)
    plt.xlabel("Classes")
    plt.ylabel("Counts")
    plt.title("Class Distribution after Rebalancing classes")
    plt.show()

    return df

In [ ]:
''' splitting encoded data, scaling, then imputing '''
def split_scale_impute(df):
    # -------------------------------------- Split ----------------------------------
    X = df.drop(columns=['sfdm2'])
    y = df[['sfdm2']]

    # split into training vs test data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=42)

    # -------------------------------------- Scale all Features ----------------------------------
    scaler = StandardScaler()
    XS_train = scaler.fit_transform(X_train)
    XS_test = scaler.transform(X_test)

    # -------------------------------------- Impute ----------------------------------
    imputer = KNNImputer(n_neighbors=10) # initialize imputer 

    # training
    XS_train = imputer.fit_transform(XS_train) # fit and transform training set 

    # testing
    XS_test = imputer.transform(XS_test)

    # -------------------------------------- NaN Check ----------------------------------
    train_df = pd.DataFrame(XS_train)
    test_df = pd.DataFrame(XS_test)

    print(f"Contains missing values xtrain: {train_df.isna().any().any()}")
    print(f"Contains missing values xtest: {test_df.isna().any().any()}")
    print(f"Contains missing values ytrain: {y_train.isna().any().any()}")
    print(f"Contains missing values ytest: {y_test.isna().any().any()}")
    return XS_test, XS_train

#### Evaluation Methods

In [ ]:
''' t-SNE Visualization '''
def TSNE_visualization(x_data, trueLabels, predLabels):
    ''' Produces two t-SNE plots, comparing the true vs predicted labels '''
    le1 = LabelEncoder()
    le2 = LabelEncoder()
    trueLabels_enc = le1.fit_transform(trueLabels)
    predLabels_enc = le2.fit_transform(predLabels)

    # plot to visualize accuracy
    T = TSNE(learning_rate='auto', init='random', perplexity=30, random_state=42)
    X_TSNE = T.fit_transform(x_data) # get points

    # compare clustering
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))

    # true labels
    ax[0].scatter(X_TSNE[:, 0], X_TSNE[:, 1], c=trueLabels_enc, cmap='cividis')
    ax[0].set_xlabel('TSNE component 1')
    ax[0].set_ylabel('TSNE component 2')
    ax[0].set_title('True Clustering')

    # predicted labels
    ax[1].scatter(X_TSNE[:, 0], X_TSNE[:, 1], c=predLabels_enc, cmap='inferno')
    ax[1].set_xlabel('TSNE component 1')
    ax[1].set_title('Predicted Clustering')

    plt.show()

    return

In [ ]:
''' Entropy Evaluation '''
# TO DO

#### 3.1 Multinomial Logistic Regression <br>

In [ ]:
class MN_Logistic_Regression_model:
    def __init__(self, 
                 num_classes: int,
                 x_train: pd.DataFrame,
                 y_train: pd.DataFrame,
                 x_test: pd.DataFrame,
                 y_test: pd.DataFrame,
                 balance_classes: bool):
        
        self.y_train = np.ravel(y_train)
        self.y_test = np.ravel(y_test)
        self.x_train = x_train
        self.x_test = x_test
        self.num_classes = num_classes

        if balance_classes == True:
            my_model = LogisticRegression(class_weight='balanced')

        else: 
            my_model = LogisticRegression()

        my_model.fit(self.x_train, self.y_train)

        # training predictions
        self.y_train_predLabels = my_model.predict(x_train)

        # testing predictions
        self.y_test_probabilities = my_model.predict_proba(x_test)
        self.y_test_predLabels = my_model.predict(x_test)

    def get_accuracies(self):
        test_acc = accuracy_score(self.y_test, self.y_test_predLabels) * 100
        train_acc = accuracy_score(self.y_train, self.y_train_predLabels) * 100
        print(f"training accuracy: {train_acc:.2f}%")
        print(f"testing accuracy: {test_acc:.2f}%")
        return
    
    def visualize_clustering(self):
        return TSNE_visualization(self.x_test, self.y_test, self.y_test_predLabels)

#### 3.2 Unsupervised Method: Leiden Clustering <br>

In [ ]:
class Leiden_clustering:
    def __init__(self, 
                 k: int,
                 x_data: pd.DataFrame,
                 y_data: pd.DataFrame,
                 resolution=1.0
                 ):
        
        nbrs = NearestNeighbors(n_neighbors=k).fit(x_data)
        distances, indices = nbrs.kneighbors(x_data)

        edges = [(i, j) for i, neigh in enumerate(indices) for j in neigh[1:]]

        My_G = nx.Graph()
        My_G.add_nodes_from(y_data)
        My_G.add_edges_from(edges)

        G = ig.Graph(n=len(x_data), edges = edges, directed = False)
        self.G = G.simplify()

        pos = nx.spectral_layout(My_G)

        self.x_data = x_data
        self.y_data = y_data

        # for single UMAP
        partition = leidenalg.find_partition(self.G, leidenalg.RBConfigurationVertexPartition, resolution_parameter = resolution)
        self.labels = np.array(partition.membership)

    def get_UMAP(self):
        ''' get 2 UMAP plots comparing the true vs predicted labels'''
        X_UMAP = umap.UMAP().fit_transform(self.x_data)
        labels = [self.y_data, self.labels]
        titles = ['True labels', 'Predicted labels']

        fig, ax = plt.subplots(1, 2, figsize=(10, 4))

        for plt_i in range(2):
            for i in range(np.max(labels[plt_i])+ 1):
                idx = np.argwhere(labels[plt_i] == i)
                ax[plt_i].scatter(X_UMAP[idx, 0], X_UMAP[idx, 1], marker = '.', alpha = 0.3, label = str(i))

            ax[plt_i].set_aspect('equal', 'datalim')
            ax[plt_i].set_title(titles[plt_i])
            ax[plt_i].legend()
        
        plt.tight_layout
        plt.show()

        return

#### 3.3 Unsupervised Method: Gaussian Mixture Models <br>

In [ ]:
class Gaussian_Mixture_model:
    def __init__(self, 
                 num_components: int,
                 rand_state: int,
                 x_data: pd.DataFrame,
                 y_data: pd.DataFrame,
                 target: str):
        
        self.num_components = num_components
        self.x_data = x_data
        self.y_data = y_data

        my_model = GaussianMixture(n_components = num_components, random_state = rand_state).fit(x_data)
    
        # extract centers and pred labels
        self.center = my_model.means_
        self.pred_label_probabilities = my_model.predict_proba(x_data)
        self.predLabels = my_model.predict(x_data)
        self.trueLabels = y_data[target].values

    def visualize_clustering(self):
        return TSNE_visualization(self.x_data, self.trueLabels, self.predLabels)

#### 3.4 Supervised Method: Naive Bayes <br>

In [ ]:
class Naive_Bayes_model:
    def __init__(self, 
                 num_classes: int,
                 x_train: pd.DataFrame,
                 y_train: pd.DataFrame,
                 x_test: pd.DataFrame,
                 y_test: pd.DataFrame,
                 update_priors: bool):
        
        self.y_train = np.ravel(y_train)
        self.y_test = np.ravel(y_test)
        self.x_train = x_train
        self.x_test = x_test
        self.num_classes = num_classes
    
        if update_priors == True:
            p = 1 / num_classes
            priors_list = [p] * num_classes
            my_model = GaussianNB(priors=priors_list)
        
        else: 
            my_model = GaussianNB()

        my_model.fit(self.x_train, self.y_train)

        # training predictions
        self.y_train_predLabels = my_model.predict(x_train)

        # testing predictions
        self.y_test_probabilities = my_model.predict_proba(x_test)
        self.y_test_predLabels = my_model.predict(x_test)

    def get_accuracies(self):
        test_acc = accuracy_score(self.y_test, self.y_test_predLabels) * 100
        train_acc = accuracy_score(self.y_train, self.y_train_predLabels) * 100
        print(f"training accuracy: {train_acc:.2f}%")
        print(f"testing accuracy: {test_acc:.2f}%")
        return
    
    def visualize_clustering(self):
        return TSNE_visualization(self.x_test, self.y_test, self.y_test_predLabels)

## 4 Evaluation: present results, compare methods
- Table comparing accuracies
- Entropy measurement
- TSNE graphs

## Discussion: interpret results, discuss model limitations and improvements

## Contribution<br>
Andy: Unsupervised modeling + fine tuning, data evaluation <br>
Anika: Data preprocessing, visualization, data evaluation <br>
Kara: Dataset selection, EDA, feature selection, limitations & improvements <br>
Yashesha: Supervised modeling + fine tuning, data evaluation <br>

## References